# nb36 - Spacetime pair-bias attention: delta-t as a graph edge

Every PairT variant so far biased attention with SPATIAL pair features only (dx, dy, dR, E-sum, E-min) - the time dimension never entered the edges. Physically, cells of one shower are time-coherent with each other while pileup cells are incoherent with the shower core, and that is a PAIRWISE statement: per-cell absolute time (already in the tokens, proven weak) cannot express it cleanly, but an edge feature dt_ij can. This is the spacetime-graph view of our winning model: nodes = cells, edges = (dx, dy, dR, Esum, dt, dt-validity), edges bias every attention head (ParT-style U), readout = the proven subtract-then-calibrate structure at 9x9.

Caveat noted up front: simulation cell times may be unsmeared (no readout resolution), so any gain here is an UPPER bound on what real ~20-50 ps timing would give. 81^2 = 6.5k pairs, ~15 MB/batch - far below the banned 240^2 regime. Anchors: nb32 W4 mean 0.0465, 3-seed ens 0.0450; 7-model ens 0.0444; targets 0.06/0.045/0.035/0.032/0.030/0.030.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd, uproot, awkward as ak
import torch, torch.nn as nn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH, EPS
MB = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
OUT = REPO / 'reports' / 'predictions'; OUT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB36_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB36_MODE', 'full')
if MODE == 'smoke': MB = MB[:8]
THRESH = 2.49
print('device', DEVICE, '| mode', MODE, '|', len(MB), 'minbias files | THRESH', THRESH, 'MeV')

device cpu | mode full | 94 minbias files | THRESH 2.49 MeV


In [2]:
TK = ['cell_x','cell_y','energy','cell_energies_front','cell_energies_back',
      'cell_times_front','cell_times_back','imodx','jmody']
AUX = ['sig_flux_prod_vertex_z','sig_flux_eTot']
def event_geom(cc):
    x, yy, e = cc['cell_x'], cc['cell_y'], cc['energy']
    ix, iy = cc['imodx'], cc['jmody']
    seed = int(np.argmax(e))
    pts = np.stack([x, yy], 1)
    pitch = np.full(len(x), np.nan)
    for key in {(int(p), int(q)) for p, q in zip(ix, iy)}:
        sel = (ix == key[0]) & (iy == key[1]); p = pts[sel]
        if len(p) >= 2:
            d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf
            pitch[sel] = np.median(np.min(d, axis=1))
    fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
    pitch[~np.isfinite(pitch)] = fill
    ps = pitch[seed]
    ei = (x - x[seed]) / ps; ej = (yy - yy[seed]) / ps
    di = np.round(ei).astype(int); dj = np.round(ej).astype(int)
    ok = (np.abs(ei - di) < 0.15) & (np.abs(ej - dj) < 0.15)
    return seed, ps, di, dj, ok
def build_grid(files, label):
    EV = []
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TK + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        et_all = ak.to_numpy(a['sig_flux_eTot']).astype(float)
        for i in np.flatnonzero((vz < 100.0) & (et_all >= 1.0) & (et_all <= 100.0)):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TK}
            e = cc['energy']
            if len(e) < 3: continue
            seed, ps, di, dj, ok = event_geom(cc)
            if ok.mean() < 0.5 or not ok[seed]: continue
            tf = cc['cell_times_front']; tb = cc['cell_times_back']
            tf = np.where(np.isfinite(tf) & (tf != 0) & (np.abs(tf) < 1e4), tf, np.nan)
            tb = np.where(np.isfinite(tb) & (tb != 0) & (np.abs(tb) < 1e4), tb, np.nan)
            EV.append(dict(di=di[ok].astype(np.int16), dj=dj[ok].astype(np.int16),
                           e=e[ok].astype(np.float32),
                           fr=cc['cell_energies_front'][ok].astype(np.float32),
                           bk=cc['cell_energies_back'][ok].astype(np.float32),
                           tf=tf[ok].astype(np.float32), tb=tb[ok].astype(np.float32),
                           ps=float(ps), reg=int(np.argmin(np.abs(PITCH - ps))),
                           Etrue=float(et_all[i])))
    print(f'{label}: {len(EV)} events')
    return EV
t0 = time.time()
ME = build_grid(MB, 'minbias')
print(f'build {time.time()-t0:.0f}s')

minbias: 72554 events
build 90s


In [3]:
def make_windows(W):
    rows = []; keep = []
    for i, ev in enumerate(ME):
        m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
        if m.sum() < 1: continue
        di, dj, e, fr, bk, tf, tb = (v[m] for v in (ev['di'], ev['dj'], ev['e'], ev['fr'], ev['bk'], ev['tf'], ev['tb']))
        t0f = np.nanmedian(tf) if np.isfinite(tf).any() else 0.0
        t0b = np.nanmedian(tb) if np.isfinite(tb).any() else 0.0
        tfc = np.where(np.isfinite(tf), tf - t0f, 0.0); htf = np.isfinite(tf).astype(np.float32)
        tbc = np.where(np.isfinite(tb), tb - t0b, 0.0); htb = np.isfinite(tb).astype(np.float32)
        rdr = np.hypot(di, dj)
        cont = np.stack([np.log1p(np.clip(e, 0, None)), np.log1p(np.clip(fr, 0, None)),
                         np.log1p(np.clip(bk, 0, None)), di.astype(np.float32), dj.astype(np.float32),
                         rdr, np.full(len(e), np.log(ev['ps'])), np.clip(tfc, -5, 5), np.clip(tbc, -5, 5)], 1)
        oh = np.zeros((len(e), len(PITCH)), np.float32); oh[:, ev['reg']] = 1.0
        tok = np.concatenate([cont, htf[:, None], htb[:, None], oh], 1).astype(np.float32)
        rows.append((tok, float(e.sum()), float(e.max()), ev['Etrue'])); keep.append(i)
    return rows, np.array(keep)
def splits_for(keep):
    remap = -np.ones(len(ME), int); remap[keep] = np.arange(len(keep))
    a, b, t = split(len(ME))
    return (remap[a][remap[a] >= 0], remap[b][remap[b] >= 0], remap[t][remap[t] >= 0])

In [4]:
CFG = dict(d=128, nhead=4, layers=3, dropout=0.1, lr=3e-4, wd=1e-4, batch=96, huber_delta=0.1, pair_hidden=32)
NG = 5; NC = 9; NR = 5
def pair_features(R):
    di, dj, le, tfc, htf = R[..., 0], R[..., 1], R[..., 2], R[..., 3], R[..., 4]
    dx = di.unsqueeze(2) - di.unsqueeze(1); dy = dj.unsqueeze(2) - dj.unsqueeze(1)
    dR = torch.sqrt(dx * dx + dy * dy + 1e-6)
    esum = le.unsqueeze(2) + le.unsqueeze(1)
    hpair = htf.unsqueeze(2) * htf.unsqueeze(1)
    dt = (tfc.unsqueeze(2) - tfc.unsqueeze(1)) * hpair
    return torch.stack([dx, dy, dR, esum, dt, hpair], -1)
class PairEmbed(nn.Module):
    def __init__(self):
        super().__init__()
        h = CFG['pair_hidden']
        self.net = nn.Sequential(nn.Linear(6, h), nn.GELU(), nn.Linear(h, h), nn.GELU(), nn.Linear(h, CFG['nhead']))
    def forward(self, pf): return self.net(pf).permute(0, 3, 1, 2).contiguous()
class PMHA(nn.Module):
    def __init__(self, d, nh, drop):
        super().__init__(); self.h = nh; self.dh = d // nh
        self.q = nn.Linear(d, d); self.k = nn.Linear(d, d); self.v = nn.Linear(d, d); self.o = nn.Linear(d, d)
        self.drop = nn.Dropout(drop)
    def forward(self, x, U, kv):
        B, L2, d = x.shape
        q = self.q(x).view(B, L2, self.h, self.dh).transpose(1, 2)
        k = self.k(x).view(B, L2, self.h, self.dh).transpose(1, 2)
        v = self.v(x).view(B, L2, self.h, self.dh).transpose(1, 2)
        s = (q @ k.transpose(-2, -1)) / (self.dh ** 0.5) + U
        s = s.masked_fill((~kv).view(B, 1, 1, L2), -1e9)
        return self.o((self.drop(s.softmax(-1)) @ v).transpose(1, 2).reshape(B, L2, d))
class Block(nn.Module):
    def __init__(self, d, nh, drop):
        super().__init__(); self.n1 = nn.LayerNorm(d); self.attn = PMHA(d, nh, drop); self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Dropout(drop), nn.Linear(4*d, d))
        self.drop = nn.Dropout(drop)
    def forward(self, x, U, kv):
        x = x + self.drop(self.attn(self.n1(x), U, kv)); return x + self.drop(self.ff(self.n2(x)))
class SpacetimeSub(nn.Module):
    def __init__(self, in_dim, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.embed = nn.Linear(in_dim, d)
        self.pair = PairEmbed()
        self.blocks = nn.ModuleList([Block(d, CFG['nhead'], CFG['dropout']) for _ in range(CFG['layers'])])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 1))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, x, m, g, ecell, R):
        U = self.pair(pair_features(R))
        h = self.embed(x)
        for blk in self.blocks: h = blk(h, U, m)
        fl = self.fhead(h).squeeze(-1)
        w = torch.sigmoid(fl) * m.float()
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        wm = m.unsqueeze(-1).float()
        p = self.norm((h * wm).sum(1) / wm.sum(1).clamp(min=1))
        return base + self.head(torch.cat([p, g], 1))
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
def prep(W):
    rows, keep = make_windows(W)
    ktr, kva, kte = splits_for(keep)
    N = len(rows); L = (2*W+1)**2; IN_DIM = rows[0][0].shape[1]
    y = np.array([np.log(max(r[3], 1e-3)) for r in rows], np.float32)
    Et = np.array([r[3] for r in rows], np.float32)
    sumE = np.array([r[1] for r in rows], np.float32)
    X = np.zeros((N, L, IN_DIM), np.float32); M = np.zeros((N, L), np.bool_)
    G = np.zeros((N, NG), np.float32); Eraw = np.zeros((N, L), np.float32)
    Rn = np.zeros((N, L, NR), np.float32)
    for i, (tok, se, sde, et) in enumerate(rows):
        n = tok.shape[0]; X[i, :n] = tok; M[i, :n] = True
        Rn[i, :n] = tok[:, [3, 4, 0, 7, 9]]
        e = np.expm1(tok[:, 0]); Eraw[i, :n] = e
        lat = float(np.sqrt((e * tok[:, 5] ** 2).sum() / (e.sum() + EPS)))
        fbr = float(np.expm1(tok[:, 1]).sum() / (np.expm1(tok[:, 2]).sum() + EPS))
        G[i] = [np.log1p(se), np.log1p(sde), np.log(n), fbr, lat]
    la0, lb0 = np.polyfit(np.log1p(0.5 * sumE[ktr]), y[ktr], 1)
    G = (G - G[ktr].mean(0)) / (G[ktr].std(0) + EPS)
    cont = X[ktr][:, :, :NC].reshape(-1, NC)[M[ktr].reshape(-1)]
    mean = cont.mean(0); std = cont.std(0) + EPS
    X[:, :, :NC] = (X[:, :, :NC] - mean) / std; X[~M] = 0.0
    T = dict(X=torch.from_numpy(X).to(DEVICE), M=torch.from_numpy(M).to(DEVICE),
             G=torch.from_numpy(G).to(DEVICE), Y=torch.from_numpy(y).unsqueeze(1).to(DEVICE),
             E=torch.from_numpy(Eraw).to(DEVICE), R=torch.from_numpy(Rn).to(DEVICE))
    print(f'W={W}: N {N}, tr/va/te {len(ktr)}/{len(kva)}/{len(kte)}, IN_DIM {IN_DIM}')
    return T, y, Et, ktr, kva, kte, IN_DIM, float(la0), float(lb0)

In [5]:
def train_eval(T, y, Et, ktr, kva, kte, IN_DIM, la0, lb0, W, seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SpacetimeSub(IN_DIM, la0, lb0).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ck = CKPT / f'nb36_pairst_s{seed}.pt'
    def batches(idx, bs, sh):
        idx = np.asarray(idx)
        if sh: idx = rng.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(b): return model(T['X'][b], T['M'][b], T['G'][b], T['E'][b], T['R'][b])
    def run(idx):
        model.eval(); out = []
        with torch.no_grad():
            for b in batches(idx, 256, False): out.append(fwd(b).cpu().numpy().ravel())
        return np.concatenate(out)
    def vloss():
        model.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256, False):
                s += nn.functional.huber_loss(fwd(b), T['Y'][b], delta=CFG['huber_delta']).item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume W{W} s{seed} from epoch {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(ktr, CFG['batch'], True):
            opt.zero_grad()
            nn.functional.huber_loss(fwd(b), T['Y'][b], delta=CFG['huber_delta']).backward()
            opt.step()
        sched.step(); vv = vloss()
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(model.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    model.load_state_dict(bstate)
    a, b2 = np.polyfit(run(kva), y[kva], 1)
    pe = np.exp(a * run(kte) + b2)
    return float(resolution(pe, Et[kte])['sigma_eff']), pe

In [6]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1, 2]}[MODE]
W = 4
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb36_pairst{TAG}.csv'
done = set()
if CSVP.exists():
    prev = pd.read_csv(CSVP); done = set(prev['seed'])
    print('resume, done:', sorted(done))
T, y, Et, ktr, kva, kte, IN_DIM, la0, lb0 = prep(W)
for seed in SEEDS:
    if seed in done: print('skip', seed); continue
    t0 = time.time()
    sig, pe = train_eval(T, y, Et, ktr, kva, kte, IN_DIM, la0, lb0, W, seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb36_pred{TAG}_pairst_s{seed}.npy', pe)
    row = dict(seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t0))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'pairst seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
RES = pd.read_csv(CSVP); print(RES.to_string(index=False))

W=4: N 72554, tr/va/te 50787/10883/10884, IN_DIM 16


  resume W4 s0 from epoch 11


pairst seed 0: sigma_eff 0.0472 (16889s)


pairst seed 1: sigma_eff 0.0462 (11993s)


pairst seed 2: sigma_eff 0.0459 (9313s)


 seed  sigma_eff  elapsed
    0     0.0472    16889
    1     0.0462    11993
    2     0.0459     9313


## Verdict: does the spacetime edge help, and does it stack with the 7-model pool?

In [7]:
print('anchors: nb32 W4 mean 0.0465 / ens 0.0450 | 7-model ens 0.0444 | targets 0.06/0.045/0.035/0.032/0.030/0.030')
print(f'pairst: {RES.sigma_eff.mean():.4f} +/- {(RES.sigma_eff.std() if len(RES) > 1 else 0):.4f} (n={len(RES)})')
te_e = Et[kte]
ps = [np.load(OUT / f'nb36_pred{TAG}_pairst_s{s}.npy') for s in SEEDS
      if (OUT / f'nb36_pred{TAG}_pairst_s{s}.npy').exists()]
def show(name, pe):
    print(f'{name}: overall {resolution(pe, te_e)["sigma_eff"]:.4f}')
    edges = np.quantile(te_e, np.linspace(0, 1, 7))
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        print(f'  E {edges[i]:6.1f}-{edges[i+1]:6.1f} GeV: {resolution(pe[mm], te_e[mm])["sigma_eff"]:.4f}')
if len(ps) >= 2: show(f'pairst {len(ps)}-seed ensemble', np.stack(ps).mean(0))
OLD = [OUT / f'nb32_pred_W4_s{s}.npy' for s in (0, 1, 2)]
OLD += [OUT / f'nb34_pred_pure_s{s}.npy' for s in (3, 4)]
OLD += [OUT / f'nb34_pred_cleanaux_s{s}.npy' for s in (0, 1)]
old = [np.load(p) for p in OLD if p.exists()]
if old and len(old[0]) == len(te_e) and ps:
    show(f'everything ({len(old) + len(ps)} models)', np.stack(old + ps).mean(0))

anchors: nb32 W4 mean 0.0465 / ens 0.0450 | 7-model ens 0.0444 | targets 0.06/0.045/0.035/0.032/0.030/0.030
pairst: 0.0464 +/- 0.0007 (n=3)
pairst 3-seed ensemble: overall 0.0452
  E    2.2-  10.7 GeV: 0.0675
  E   10.7-  17.4 GeV: 0.0496
  E   17.4-  24.0 GeV: 0.0372
  E   24.0-  34.1 GeV: 0.0390
  E   34.1-  53.1 GeV: 0.0375
  E   53.1- 100.0 GeV: 0.0392
everything (10 models): overall 0.0443
  E    2.2-  10.7 GeV: 0.0659
  E   10.7-  17.4 GeV: 0.0502
  E   17.4-  24.0 GeV: 0.0367
  E   24.0-  34.1 GeV: 0.0374
  E   34.1-  53.1 GeV: 0.0368
  E   53.1- 100.0 GeV: 0.0376
